This notebook and saves an ortho on the shendure data as preprocessed in `preprocessing` & demonstrates that estimated nb parameters track closely with UMI means, as expected.


# Setup

In [1]:
#imports
import pandas as pd
import numpy as np
import time
import pickle
from pathlib import Path
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt
import os
import signal
from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm

In [2]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client, performance_report

local=True
if not local:
    worker_cores = 1
    worker_memory = "96G"
    worker_processes = 1
    cluster_jobs = 16

    cluster=SLURMCluster(
        cores=worker_cores,#cores per slurm job
        memory=worker_memory,#memory per slurm job
        processes=worker_processes,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=2-23:40:00",
            f"--output=worker_%j.out"]
    )

    cluster.scale(jobs=cluster_jobs)

    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )
else:
    from dask.distributed import Client, LocalCluster
    cluster=LocalCluster()
    client = Client(cluster)

In [3]:
client.dashboard_link

'http://127.0.0.1:8787/status'

# Describe with an ortho

Set paths

In [4]:
#data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"
data_root="/vast/palmer/pi/reilly/tabula_data"
path=f"{data_root}/shendure"
name="shendure_ortho_consider_missing_20260320"
report_path = Path(name + "_dask_performance_report.html")

Next, clean the data format

In [5]:
shendure_data_raw=pd.read_csv(f"{path}/shendure_counts_not_grouped.txt",sep="\t")

In [6]:
cols=list(scm.MPRA_UMIWISE_ALLOWED)
cols.remove("reads_DNA")
cols

['reads_transfection_bc',
 'mpra_bc',
 'cre_id',
 'cell_bc',
 'transfection_bc',
 'cell_type',
 'umis_transfection_bc',
 'umis_mpra_bc',
 'rep_id',
 'reads_mpra_bc']

In [7]:
shendure_data_reduced=shendure_data_raw[cols]
shendure_data_reduced.to_csv(f"{path}/shendure_processed.tsv",sep="\t", index=False)

In [ ]:
try:
    with performance_report(filename=str(report_path)):
        if os.path.isdir(path+"/"+name):
            print("[+] Model found. Loading...")
            primordial=scm.ortho.load(client,path,name)
            shendure=primordial.training_data
        else:
            print("[+] Model not found. Creating...")

            #load data
            shendure=scm.scMPRA_data.from_tsv(f"{path}/shendure_processed.tsv")
            shendure.set_negative_controls(["minP","noP"])
            shendure.set_reference_cell("Pluripotent")
            shendure.ortho_filter()
            
            shendure.set_consider_missing(True)

            primordial=scm.ortho()
            primordial.criss_cross(client=client,
                               dat=shendure)
            primordial.extract_params(client)
            primordial.save(path,name,client=client)
finally:
    print("! Done, shutting down",flush=True)
    time.sleep(10)
    client.close()
    cluster.close()
    time.sleep(10)

# Examine QC metrics

In [ ]:
primordial.compute_model_qc()

In [ ]:
by_cre_thetas=[]
for key in primordial.by_cre.model:
    by_cre_thetas.append(primordial.by_cre.model[key].result()["weights"]["theta"].squeeze())
by_cre_thetas=np.array(by_cre_thetas)

by_cell_type_thetas=[]
for key in primordial.by_cell_type.model:
    by_cell_type_thetas.append(primordial.by_cell_type.model[key].result()["weights"]["theta"].squeeze())
by_cell_type_thetas=np.array(by_cell_type_thetas)


In [ ]:
sns.violinplot(by_cre_thetas)

In [ ]:
sns.violinplot(by_cell_type_thetas)

In [ ]:
np.mean(by_cre_thetas)

In [ ]:
np.mean(by_cell_type_thetas)

In [ ]:
np.mean(np.concatenate((by_cre_thetas,by_cell_type_thetas)))

Let's look at the mu min & max : none should be below zero, none should be above 1000.

In [ ]:
def minimax(QC):
    x=[]
    for level in QC.keys():
        if QC[level]["success"]:
            x.append(QC[level]["dat"])
    x=pd.concat(x)
    print(f"min {min(x['mu'])}, max {max(x['mu'])}")

print("cre")
minimax(primordial.by_cre_qc)
print("ct")
minimax(primordial.by_cell_qc)

All in the right ballpark!

Now let's look at the correlations.

In [ ]:
r=[]
for QC in [primordial.by_cre_qc,primordial.by_cell_qc]:
    print("---")
    
    for level in QC.keys():
        if QC[level]["success"]:
            if np.isnan(QC[level]["r_value"]):
                print(f"nan in {level}")
            else:
                if QC[level]["r_value"]<0.8:
                    print(f"low r {level}")
                r.append(QC[level]["r_value"])


sns.violinplot(r)

print(r)
#print(np.mean(r))
#for cell_type in QC:
#    print(f"{cell_type} : r={QC[cell_type]['r_value']}, slope={QC[cell_type]['slope']}")

Correlations generally look pretty good. Let's examine the cases where they aren't.

Notably all of the bad ones are from the set of "by cell-type" models.

I'm going to guess these are low-expressing CREs. Let's take a look.

In [ ]:
highlighted_cre_ids=["Cdk5r1_chr11_12595","Lamb1_chr12_2206","Lamc1_chr1_12183","Txndc12_chr4_7969"]

grouped = (
    shendure.data.compute()
    .groupby(["cre_id", "cell_type"])["umis_mpra_bc"]
    .mean()
    .reset_index()
)

# Split base vs. highlighted
base = grouped[~grouped["cre_id"].isin(highlighted_cre_ids)]
highlighted = grouped[grouped["cre_id"].isin(highlighted_cre_ids)]

# Set up plot
plt.figure(figsize=(10, 6))

# Plot base layer with jitter
sns.stripplot(
    data=base,
    x="cell_type",
    y="umis_mpra_bc",
    color="lightgray",
    jitter=0.35,
    label="Other CREs",
    size=6
)

# Overlay highlighted CREs with jitter and custom color
palette = sns.color_palette("tab10", n_colors=len(highlighted_cre_ids))
for i, cre in enumerate(highlighted_cre_ids):
    sns.stripplot(
        data=highlighted[highlighted["cre_id"] == cre],
        x="cell_type",
        y="umis_mpra_bc",
        color=palette[i],
        jitter=0.35,
        label=cre,
        size=6
    )

# Y-axis and aesthetics
plt.ylim(-1, 1)
plt.xlabel("Cell Type")
plt.ylabel("Mean UMIs (mpra_bc)")
plt.title("Mean UMIs per (cre_id, cell_type)")
plt.xticks(rotation=45)
plt.legend(title="Highlighted CREs")
plt.tight_layout()
plt.show()


In [ ]:
primordial.by_cre_qc["Cdk5r1_chr11_12595"]

In [9]:
cluster.close()
client.close()